# Introducción a OSMnx: consultar, simplificar, visualizar y guardar

Autor: [Geoff Boeing](https://geoffboeing.com/)

  - [Documentación](https://osmnx.readthedocs.io/)
  - [Artículo y datos de citación](https://doi.org/10.1111/gean.70009)
  - [Repositorio de código](https://github.com/gboeing/osmnx)
  - [Galería de ejemplos](https://github.com/gboeing/osmnx-examples)
  
Después de revisar el [notebook de demostración de funciones](00-osmnx-features-demo.ipynb), este notebook muestra con más detalle cómo consultar límites de lugares y redes viales, visualizarlos y guardar los modelos en disco.

In [ ]:
from pathlib import Path

import geopandas as gpd
import osmnx as ox

ox.__version__

OSMnx se configura con el módulo `settings`. En la [documentación](https://osmnx.readthedocs.io/en/stable/osmnx.html#module-osmnx.settings) están los parámetros configurables. Por ejemplo, por defecto OSMnx guarda en caché todas las respuestas del servidor para no consultarlo repetidamente con la misma solicitud cada vez que se ejecuta el código. Esto hace el código más rápido en ejecuciones posteriores y nos ayuda a ser "buenos vecinos" del servidor. Pero la caché se puede desactivar (o volver a activar) con el parámetro `use_cache`.

In [ ]:
# turn response caching off
ox.settings.use_cache = False

# volver a activarla y activar o desactivar el registro en la consola
ox.settings.use_cache = True
ox.settings.log_console = False

## Parte 1: obtener límites de lugares de OpenStreetMap

OSMnx permite descargar las geometrías de los límites de lugares de OpenStreetMap, proyectarlas y graficarlas. Para una demostración más detallada de las consultas por lugar, ver [este notebook](03-graph-place-queries.ipynb).

In [ ]:
# obtener el polígono del límite de Manhattan, proyectarlo y graficarlo
city = ox.geocoder.geocode_to_gdf("Manhattan, New York, USA")
city_proj = ox.projection.project_gdf(city)
ax = city_proj.plot(fc="gray", ec="none")
_ = ax.axis("off")

In [ ]:
# obtener los polígonos de los límites de varias ciudades, guardarlos como GeoPackage, proyectarlos a UTM y graficarlos
place_names = [
    "Berkeley, California, USA",
    "Oakland, California, USA",
    "Piedmont, California, USA",
    "Emeryville, California, USA",
    "Alameda, Alameda County, CA, USA",
]
east_bay = ox.geocoder.geocode_to_gdf(place_names)
Path("data").mkdir(parents=True, exist_ok=True)
east_bay.to_file("./data/east_bay.gpkg", driver="GPKG")
east_bay = ox.projection.project_gdf(east_bay)
ax = east_bay.plot(fc="gray", ec="none")
_ = ax.axis("off")

In [ ]:
# si se conoce el ID de OSM de los lugares que se quieren, se pueden consultar directamente
ox.geocoder.geocode_to_gdf(["R357794", "N8170768521", "W427818536"], by_osmid=True)

## Parte 2: descargar y modelar redes viales

OSMnx permite descargar datos de redes viales y construir redes con la topología corregida, proyectarlas y graficarlas, y guardarlas como SVG, archivos GraphML o GeoPackages para usarlas después. Las redes viales son dirigidas y conservan el sentido de las vías de un solo sentido. Para una demostración más detallada de la creación de redes viales, ver [este notebook](03-graph-place-queries.ipynb).

Una red vial se puede descargar pasando a OSMnx cualquiera de los siguientes elementos (ver los ejemplos abajo):
  - un rectángulo envolvente
  - un punto de latitud y longitud más una distancia
  - una dirección más una distancia
  - el nombre de un lugar o una lista de nombres (para geocodificarlos automáticamente y obtener sus límites)
  - un polígono con los límites de la red vial deseada
  - un archivo xml en formato .osm

También se pueden indicar distintos tipos de red:
  - 'all': descarga todas las calles y caminos de OSM, incluidos los de acceso privado (es el tipo de red predeterminado si no se indica otro)
  - 'all_public': descarga todas las calles y caminos de OSM que no son privados
  - 'bike': obtiene todas las calles y caminos que pueden usar los ciclistas
  - 'drive': obtiene las calles públicas transitables en vehículo (sin vías de servicio)
  - 'drive_service': obtiene las calles transitables en vehículo, incluidas las vías de servicio
  - 'walk': obtiene todas las calles y caminos que pueden usar los peatones

Si se quiere un grafo completamente bidireccional, basta con configurar `bidirectional_network_types` en el módulo `settings` antes de crear el grafo (por defecto incluye el tipo de red "walk").

 #### Método 1: pasar un rectángulo envolvente ####
 Construye la red con todos los nodos y vías de OSM dentro del rectángulo envolvente.

In [ ]:
# define a bounding box in San Francisco as (left, bottom, right, top)
bbox = -122.43, 37.78, -122.41, 37.79

# create network from that bounding box
G = ox.graph.graph_from_bbox(bbox, network_type="drive_service")

#### Método 2: pasar un punto de latitud y longitud y una distancia del rectángulo envolvente en metros
Crea un rectángulo envolvente a *n* metros al norte, sur, este y oeste del punto, y construye la red con todos los nodos y vías de OSM dentro de él.

In [ ]:
# definir un punto en la esquina de California St y Mason St en San Francisco
location_point = (37.791427, -122.410018)

# crear la red a partir del punto, dentro de un rectángulo de 750 m al N, S, E y O del punto
G = ox.graph.graph_from_point(location_point, dist=750, dist_type="bbox", network_type="drive")

#### Método 3: pasar un punto de latitud y longitud y una distancia *sobre la red* en metros ####

Crea un rectángulo envolvente a *n* metros al norte, sur, este y oeste del punto y construye la red con todos los nodos y vías de OSM dentro de él. Luego recorta la red eliminando todos los nodos que estén a más de *n* metros del punto a lo largo de la red.

In [ ]:
# el mismo punto, pero creando la red solo con los nodos a menos de 500 m del punto a lo largo de la red
G = ox.graph.graph_from_point(location_point, dist=500, dist_type="network")
fig, ax = ox.plot.plot_graph(G, node_color="r")

*Nota*: la gráfica anterior muestra la red a menos de 500 m (distancia de recorrido a lo largo de la red) de `location_point`. Por defecto, el parámetro `network_type` vale 'all', es decir, no se filtran los caminos que restringen ciertos tipos de tráfico. Esto también significa que las vías de un solo sentido se respetan y no se pueden recorrer en contravía. Así, los 500 m incluyen solo los nodos alcanzables en 500 m viajando en el sentido permitido de cada vía. En cambio (abajo), se puede indicar `network_type='walk'` para construir la red solo con los caminos por los que se puede caminar. Esto también hace bidireccional cada camino de la red dirigida, porque se puede caminar en cualquier dirección por el andén de una vía de un solo sentido. Así, los 500 m incluyen los nodos alcanzables en 500 m viajando en cualquier dirección (aunque sea una vía de un solo sentido).

In [ ]:
# crear la red solo con los nodos a menos de 500 m caminando a lo largo de la red desde el punto
G = ox.graph.graph_from_point(location_point, dist=500, dist_type="network", network_type="walk")
fig, ax = ox.plot.plot_graph(G, node_color="r")

#### Método 4: pasar una dirección y una distancia (*rectángulo envolvente* o *sobre la red*) en metros ####
Geocodifica la dirección, crea un rectángulo envolvente, descarga la red y luego la recorta según la distancia sobre la red (si distance_type='network').

In [ ]:
# red a partir de una dirección, incluyendo solo los nodos a menos de 1 km de la dirección a lo largo de la red
G = ox.graph.graph_from_address(
    address="350 5th Ave, New York, NY",
    dist=1000,
    dist_type="network",
    network_type="drive",
)

# se puede proyectar la red a UTM (la zona se calcula automáticamente)
G_projected = ox.projection.project_graph(G)

#### Método 5: pasar el nombre de un lugar ####

Geocodifica el nombre del lugar, obtiene el polígono de su límite y su rectángulo envolvente, descarga la red dentro del rectángulo y luego la recorta al polígono del límite del lugar.

In [ ]:
# crear la red vial dentro de los límites de la ciudad de Piedmont
G = ox.graph.graph_from_place("Piedmont, California, USA", network_type="drive")

In [ ]:
# también se pueden pasar varios lugares como una lista mixta de textos y/o diccionarios
places = [
    "Los Altos, California, USA",
    {"city": "Los Altos Hills", "state": "California"},
    "Loyola, California",
]
G = ox.graph.graph_from_place(places, truncate_by_edge=True)

In [ ]:
# guardar en disco como archivo GeoPackage y luego graficar
ox.io.save_graph_geopackage(G)
fig, ax = ox.plot.plot_graph(G, node_size=0, edge_color="w", edge_linewidth=0.2)

#### Método 6: pasar un polígono ####

Este ejemplo carga desde un Shapefile el polígono del [Mission District](http://www.zillow.com/howto/api/neighborhood-boundaries.htm), descarga la red dentro de su rectángulo envolvente y luego elimina todos los nodos que quedan fuera del polígono del lugar.

In [ ]:
calif = gpd.read_file("input_data/ZillowNeighborhoods-CA")
mission_district = calif[(calif["CITY"] == "San Francisco") & (calif["NAME"] == "Mission")]
polygon = mission_district["geometry"].iloc[0]

G2 = ox.graph.graph_from_polygon(polygon, network_type="drive_service")

#### Method #7, load a .osm xml file

In [ ]:
# create graph from .osm extract file
G = ox.graph.graph_from_xml("./input_data/West-Oakland.osm.bz2")

## Parte 3: simplificar la topología de la red vial

Normalmente OSMnx hace la simplificación de forma automática e interna, pero podemos separarla para ver cómo funciona. Los nodos de OpenStreetMap son peculiares: incluyen las intersecciones, pero también todos los puntos a lo largo de una cuadra donde la calle se curva. Estos últimos no son nodos en el sentido de la teoría de grafos, así que se eliminan algorítmicamente y el conjunto de aristas entre los nodos "verdaderos" de la red se consolida en una sola arista. Hay dos modos de simplificación, estricto y no estricto. La diferencia principal es que el modo no estricto permite simplificar a un "grafo de expansión" (es decir, si el grafo fuera no dirigido, nodos de grado 2 siempre que las aristas incidentes tengan distintos ID de OSM). Para una demostración más detallada de la simplificación topológica con OSMnx, ver [este notebook](04-simplify-graph-consolidate-nodes.ipynb).

In [ ]:
# crear una red alrededor de un punto (lat, lng) sin simplificarla todavía
location_point = (33.299896, -111.831638)
G = ox.graph.graph_from_point(
    location_point, network_type="drive_service", dist=500, simplify=False
)

In [ ]:
# desactivar el modo estricto y ver en amarillo qué nodos se eliminarían
nc = ["r" if ox.simplification._is_endpoint(G, node, None, None) else "y" for node in G.nodes()]
fig, ax = ox.plot.plot_graph(G, node_color=nc)

Los puntos de arriba son nodos de OSM. Eliminaremos los amarillos, porque no son nodos reales de la red (intersecciones o calles ciegas).

In [ ]:
# simplify the network
G = ox.simplification.simplify_graph(G)
fig, ax = ox.plot.plot_graph(G, node_color="r")

In [ ]:
# mostrar la red simplificada con las aristas coloreadas según su longitud
ec = ox.plot.get_edge_colors_by_attr(G, attr="length", cmap="plasma_r")
fig, ax = ox.plot.plot_graph(
    G, node_color="w", node_edgecolor="k", node_size=50, edge_color=ec, edge_linewidth=3
)

In [ ]:
# highlight all parallel (multiple) edges
ec = ["gray" if k == 0 or u == v else "r" for u, v, k in G.edges(keys=True)]
fig, ax = ox.plot.plot_graph(
    G, node_color="w", node_edgecolor="k", node_size=50, edge_color=ec, edge_linewidth=3
)

In [ ]:
# resaltar todas las aristas de un solo sentido en la red del Mission District de antes
ec = ["r" if data["oneway"] else "w" for u, v, key, data in G2.edges(keys=True, data=True)]
fig, ax = ox.plot.plot_graph(G2, node_size=0, edge_color=ec, edge_linewidth=1.5, edge_alpha=0.7)

## Parte 4: guardar las redes en disco

Para más ejemplos de cómo guardar y cargar redes en disco, ver [este notebook](05-save-load-networks.ipynb).

In [ ]:
# guardar la red vial como GeoPackage para trabajar en un SIG
ox.io.save_graph_geopackage(G, filepath="./data/network.gpkg")

In [ ]:
# guardar la red vial como archivo GraphML para trabajar después en OSMnx, networkx o gephi
ox.io.save_graphml(G, filepath="./data/network.graphml")

## Part 5: calculate basic network indicators

In [ ]:
# calculate basic street network metrics and display average circuity
stats = ox.stats.basic_stats(G)
stats["circuity_avg"]

En esta red vial, las calles son cerca de un 14 % más sinuosas de lo que serían los recorridos en línea recta.

Para ejemplos de análisis de redes viales, ver [este ejemplo](06-stats-indicators-centrality.ipynb).